# NERP LTMP Fish Zoning — Dataset Meet & Greet

This notebook provides a quick, thorough overview of the **NERP LTMP fish visual census zoning dataset**
(local file: `nerp_ltmp_fish_zoning.csv`). It summarizes structure, coverage, missingness, and
species abundance patterns, and prepares a wide-format matrix for modeling.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except Exception:
    sns = None

pd.set_option('display.max_columns', 200)
pd.set_option('display.max_rows', 200)


In [ ]:
DATA_PATH = Path.cwd() / 'nerp_ltmp_fish_zoning.csv'
df = pd.read_csv(DATA_PATH)
df.head()


## Structure at a glance
Each row is a species observation for a reef/site/transect/date.
Key fields: `REEF_NAME`, `SITE_NO`, `TRANSECT_NO`, `SAMPLE_DATE`, `SPECIES`, `ABUNDANCE`.


In [ ]:
df.shape

df.columns


In [ ]:
df.info()


## Missingness


In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0]


## Basic normalization and parsing
Convert numeric fields and parse dates for time coverage.


In [ ]:
df['SAMPLE_DATE'] = pd.to_datetime(df['SAMPLE_DATE'], errors='coerce')
df['ABUNDANCE'] = pd.to_numeric(df['ABUNDANCE'], errors='coerce')
df['REEF_LAT'] = pd.to_numeric(df['REEF_LAT'], errors='coerce')
df['REEF_LONG'] = pd.to_numeric(df['REEF_LONG'], errors='coerce')
df['LENGTH'] = pd.to_numeric(df['LENGTH'], errors='coerce')

df[['SAMPLE_DATE','ABUNDANCE','REEF_LAT','REEF_LONG','LENGTH']].describe(include='all')


## Survey coverage
Surveys are uniquely identified by reef/site/transect/date.


In [ ]:
survey_key = df[['REEF_NAME','SITE_NO','TRANSECT_NO','SAMPLE_DATE']].astype(str).agg('|'.join, axis=1)
summary = {
    'rows': len(df),
    'unique_species': df['SPECIES'].nunique(dropna=True),
    'unique_reefs': df['REEF_NAME'].nunique(dropna=True),
    'unique_sites': df[['REEF_NAME','SITE_NO']].drop_duplicates().shape[0],
    'unique_transects': df[['REEF_NAME','SITE_NO','TRANSECT_NO']].drop_duplicates().shape[0],
    'unique_surveys': survey_key.nunique(),
}
pd.Series(summary)


In [ ]:
date_summary = df['SAMPLE_DATE'].agg(['min','max'])
date_summary


In [ ]:
df['year'] = df['SAMPLE_DATE'].dt.year
year_counts = df['year'].value_counts().sort_index()
year_counts


In [ ]:
if len(year_counts) > 0:
    year_counts.plot(kind='bar', figsize=(10, 3), title='Observations per Year')
    plt.ylabel('Rows')
    plt.tight_layout()


## Species abundance overview


In [ ]:
abundance_summary = df['ABUNDANCE'].describe()
abundance_summary


In [ ]:
df['ABUNDANCE'].hist(bins=30, figsize=(6, 3))
plt.title('Distribution of ABUNDANCE values')
plt.xlabel('ABUNDANCE')
plt.tight_layout()


In [ ]:
top_species = (
    df.groupby('SPECIES')['ABUNDANCE']
    .sum()
    .sort_values(ascending=False)
    .head(15)
)
top_species


In [ ]:
if len(top_species) > 0:
    top_species.sort_values().plot(kind='barh', figsize=(8, 4), title='Top Species by Total Abundance')
    plt.xlabel('Total Abundance')
    plt.tight_layout()


## Per-survey richness (species count)


In [ ]:
per_survey_richness = df.groupby(survey_key)['SPECIES'].nunique()
per_survey_richness.describe()


In [ ]:
per_survey_richness.hist(bins=20, figsize=(6, 3))
plt.title('Species richness per survey')
plt.xlabel('Unique species per survey')
plt.tight_layout()


## Zoning context
Compare observation counts and mean abundance by zone flag.


In [ ]:
zone_summary = df.groupby('OPENORCLOSED_AFTER2004')['ABUNDANCE'].agg(['count','sum','mean'])
zone_summary


## Duplicates
Duplicate species records for the same reef/site/transect/date can inflate counts.


In [ ]:
dup_mask = df.duplicated(subset=['REEF_NAME','SITE_NO','TRANSECT_NO','SAMPLE_DATE','SPECIES'])
dup_mask.sum()


## Wide-format matrix for modeling
This pivots to a survey-by-species matrix (fill missing species with 0).


In [ ]:
wide = df.pivot_table(
    index=['REEF_NAME','SITE_NO','TRANSECT_NO','SAMPLE_DATE'],
    columns='SPECIES',
    values='ABUNDANCE',
    aggfunc='sum',
    fill_value=0,
)
wide.shape


In [ ]:
wide.head()


## Notes for next steps
- Use `wide` as the input matrix for anomaly detection or community modeling.
- Consider filtering rare species or surveys with very low richness if needed.
- If time trends matter, group by year/season or include `SAMPLE_DATE` as a feature.
